# LLM Adaptation Workflow
### Notebook 03 — Retrieval-Augmented Generation (RAG)

---

## What is RAG and why does it matter?

**RAG (Retrieval-Augmented Generation)** is a technique that gives a language model access to a knowledge base at query time — without changing the model's weights.

Here's the idea:
1. A user asks a question
2. We search a database of documents for the most relevant passages
3. We include those passages in the model's prompt as context
4. The model answers using that context

This solves a fundamental problem: the model's knowledge is frozen at training time. With RAG, you can update your knowledge base without retraining the model.

---

## RAG vs Fine-tuning — when to use which?

| Scenario | RAG | Fine-tuning |
|----------|-----|-------------|
| Knowledge changes frequently | ✓ Best | ✗ Retraining needed |
| Large document corpus (1000s of docs) | ✓ Scales naturally | ✗ High training cost |
| Need to cite specific sources | ✓ Built-in | ✗ Hard to guarantee |
| Specialised writing style / tone | ✗ Model style unchanged | ✓ Learns your style |
| Domain-specific reasoning patterns | ✗ Limited | ✓ Bakes reasoning in |
| Low latency inference | ✗ Retrieval adds latency | ✓ Faster at inference |
| Minimal setup time | ✓ No training needed | ✗ Hours of training |

In practice, many production systems **combine both**: fine-tune for style and reasoning, then add RAG for up-to-date factual knowledge.

---

## How RAG works — the technical picture

```
         Documents
             │
    Embedding model converts
    each chunk → vector
             │
             ▼
    ┌──────────────────┐
    │   Vector Store   │  ← Stores all chunk vectors
    │   (FAISS index)  │
    └────────┬─────────┘
             │
    At query time:
    Query → embedding → find similar vectors → retrieve top-k chunks
             │
             ▼
    [Query + retrieved chunks] → LLM → Answer
```

The key component is the **embedding model** — a model that converts text into a dense vector (list of numbers) where similar texts produce similar vectors. This lets us find relevant documents using mathematical similarity rather than keyword matching.

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path

from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

PROCESSED_DIR = Path("../data/processed")

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---

## 1. Embedding model — converting text to vectors

An **embedding model** is a neural network trained specifically to encode meaning into vectors. Unlike an LLM which generates text, an embedding model just outputs a list of numbers.

The most important property: **semantically similar texts produce similar vectors**.

We use `all-MiniLM-L6-v2` — a small, fast, high-quality embedding model from Sentence Transformers.

In [ ]:
# Load the embedding model
# This is a 22M parameter model — much smaller than LLMs, designed purely for encoding
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Move to MPS if available
if device == "mps":
    embed_model = embed_model.to(device)

print("Embedding model ready.")
print(f"Output dimension: {embed_model.get_sentence_embedding_dimension()} (each text → a vector of this length)")

In [ ]:
# Let's see embeddings in action — notice how similar texts produce similar vectors

sentences = [
    "Apple reported strong revenue growth in the fiscal year.",
    "The company's annual sales increased significantly.",          # similar meaning
    "JPMorgan expanded its investment banking operations.",         # different topic
    "Revenue for the year showed impressive year-over-year gains." # similar meaning
]

embeddings = embed_model.encode(sentences, convert_to_numpy=True)

print(f"Each sentence → vector of shape: {embeddings[0].shape}")
print(f"(384 numbers representing the semantic meaning of the text)")
print()

# Compute cosine similarity between all pairs
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(embeddings)

print("Cosine similarity matrix (1.0 = identical meaning, 0.0 = unrelated):")
print()
labels = ["Apple revenue", "Annual sales up", "JPMorgan banking", "Revenue gains"]
header = f"{'':20s}" + "".join(f"{l:18s}" for l in labels)
print(header)
for i, label in enumerate(labels):
    row = f"{label:20s}" + "".join(f"{sim[i][j]:18.3f}" for j in range(len(labels)))
    print(row)

print()
print("Notice: 'Apple revenue' and 'Revenue gains' score highly (similar meaning),")
print("but 'JPMorgan banking' scores low against the revenue sentences (different topic).")

---

## 2. Building the vector store

A **vector store** (or **vector database**) stores embeddings and lets us find the most similar ones to a query. We use **FAISS** (Facebook AI Similarity Search) — a fast, production-grade library for this.

Think of it like a search engine for meaning rather than keywords.

In [ ]:
# Load our document examples from Notebook 02
# In a real system, these would be all the chunks from your document corpus

examples_path = PROCESSED_DIR / "examples.json"
if examples_path.exists():
    with open(examples_path) as f:
        examples = json.load(f)
    # Use the context field as our knowledge base
    documents = [ex["context"] for ex in examples if "context" in ex]
else:
    # Fallback: use hardcoded examples if notebook 02 hasn't been run
    documents = [
        "Apple Inc. reported total net sales of $383.3 billion for the fiscal year ended September 30, 2023, compared to $394.3 billion for fiscal 2022, representing a decrease of 2.8%.",
        "Apple's Services segment generated $85.2 billion in revenue, driven by the App Store, Apple Music, iCloud, and Apple TV+. This represented a 16% increase over the prior year.",
        "Microsoft reported revenue of $211.9 billion for fiscal year 2023, up 7% year-over-year. Operating income was $88.5 billion and net income was $72.4 billion.",
        "Microsoft's Intelligent Cloud segment, which includes Azure, generated revenue of $87.9 billion, an increase of 19% year-over-year, driven by Azure's 28% growth.",
        "JPMorgan Chase reported net income of $49.6 billion for fiscal year 2023, driven by higher net interest income as interest rates rose throughout the year.",
        "JPMorgan Chase manages credit risk through diversification, credit limits, collateral requirements, and credit derivatives. The firm uses internal credit ratings and stress testing.",
        "Apple's iPhone revenue was $200.6 billion in fiscal 2023, representing 52% of total company revenue. International markets accounted for 58% of total Apple revenues.",
        "Microsoft Azure and other cloud services revenue grew 27% in fiscal 2023. The company signed major AI partnerships and integrated OpenAI technology across its product suite.",
    ]

print(f"Loaded {len(documents)} document chunks for the knowledge base.")
print(f"\nExample document:")
print(f"  {documents[0]}")

In [ ]:
# Step 1: Embed all documents
print("Embedding documents...")
doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(f"\nEmbedding matrix shape: {doc_embeddings.shape}")
print(f"({len(documents)} documents × {doc_embeddings.shape[1]} dimensions)")  

In [ ]:
# Step 2: Build FAISS index
# FAISS stores the vectors and supports fast nearest-neighbour search

embedding_dim = doc_embeddings.shape[1]  # 384 for this model

# IndexFlatIP = exact inner-product (cosine) search
# For large corpora (millions of documents), use approximate indexes like IndexIVFFlat
index = faiss.IndexFlatIP(embedding_dim)

# Normalise vectors before adding (turns inner product into cosine similarity)
faiss.normalize_L2(doc_embeddings)
index.add(doc_embeddings.astype(np.float32))

print(f"FAISS index built.")
print(f"Index type    : {type(index).__name__}")
print(f"Vectors stored: {index.ntotal}")
print()
print("The index can now find the most semantically similar documents")
print("to any query in milliseconds, even at millions of documents.")

---

## 3. Retrieval — finding relevant documents

In [ ]:
def retrieve(query: str, k: int = 3) -> list[dict]:
    """
    Given a query, find the k most relevant documents.
    Returns a list of {text, score} dicts.
    """
    # Embed the query using the same model
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Search the index — returns distances and indices of top-k results
    scores, indices = index.search(query_embedding.astype(np.float32), k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": documents[idx],
            "score": float(score),
        })
    return results


# Test retrieval
query = "How did Apple's revenue compare to the previous year?"
results = retrieve(query, k=3)

print(f"Query: {query}\n")
print("Top retrieved documents:")
print("-" * 60)
for i, r in enumerate(results, 1):
    print(f"[{i}] Similarity: {r['score']:.3f}")
    print(f"    {r['text']}")
    print()

In [ ]:
# Test with a different query to see retrieval working across topics
query2 = "What cloud services does Microsoft offer and how are they performing?"
results2 = retrieve(query2, k=3)

print(f"Query: {query2}\n")
print("Top retrieved documents:")
print("-" * 60)
for i, r in enumerate(results2, 1):
    print(f"[{i}] Similarity: {r['score']:.3f}")
    print(f"    {r['text']}")
    print()

---

## 4. Building the RAG pipeline

Now we connect retrieval to generation. The pattern:

1. Retrieve relevant chunks for the query
2. Build a prompt that includes those chunks as context
3. Ask the LLM to answer using only that context

This last step — "answer using only the provided context" — is crucial. It's what prevents the model from hallucinating facts not in your documents.

In [ ]:
# Load the LLM for generation
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("LLM loaded.")

In [ ]:
def rag_answer(question: str, k: int = 3, max_new_tokens: int = 200) -> str:
    """
    Full RAG pipeline: retrieve → augment prompt → generate answer.
    """
    # Step 1: Retrieve relevant documents
    retrieved = retrieve(question, k=k)
    context = "\n\n".join([f"Source {i+1}: {r['text']}" for i, r in enumerate(retrieved)])
    
    # Step 2: Build augmented prompt
    # We explicitly instruct the model to use ONLY the provided context
    system = (
        "You are a financial analyst assistant. "
        "Answer the question using ONLY the information provided in the context below. "
        "If the context doesn't contain enough information, say so clearly."
    )
    
    user_message = f"""Context:
{context}

Question: {question}"""
    
    # Step 3: Format as chat and generate
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_message},
    ]
    formatted = tokeniser.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokeniser(formatted, return_tensors="pt").to(llm.device)
    
    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,    # Lower temperature = more factual, less creative
            do_sample=True,
            pad_token_id=tokeniser.eos_token_id,
        )
    
    response = tokeniser.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response, retrieved


# Run the full RAG pipeline
question = "How did Apple's revenue perform in 2023?"
answer, sources = rag_answer(question)

print(f"Question: {question}")
print()
print("Retrieved sources:")
for i, s in enumerate(sources, 1):
    print(f"  [{i}] {s['text'][:80]}... (score: {s['score']:.3f})")
print()
print("RAG Answer:")
print("-" * 60)
print(answer)

---

## 5. Measuring retrieval quality

How do we know if retrieval is working well? We measure **recall@k** — the fraction of test questions for which the relevant document appears in the top-k results.

In [ ]:
# Evaluation set: questions and the document index that should be retrieved
eval_set = [
    {"query": "What was Apple's total revenue in 2023?", "relevant_idx": 0},
    {"query": "How did Microsoft Azure grow?", "relevant_idx": 3},
    {"query": "What was JPMorgan's net income?", "relevant_idx": 4},
    {"query": "What is Apple's iPhone revenue?", "relevant_idx": 6},
]

def recall_at_k(eval_set, k=3):
    hits = 0
    for item in eval_set:
        results = retrieve(item["query"], k=k)
        retrieved_texts = [r["text"] for r in results]
        # Check if the relevant document text appears in top-k
        target = documents[item["relevant_idx"]]
        if target in retrieved_texts:
            hits += 1
    return hits / len(eval_set)

for k in [1, 3, 5]:
    score = recall_at_k(eval_set, k=k)
    print(f"Recall@{k}: {score:.0%}  ({int(score*len(eval_set))}/{len(eval_set)} questions)")

---

## Summary

In this notebook we:
- Understood what RAG is and when to use it vs fine-tuning
- Loaded an embedding model that converts text to semantic vectors
- Built a FAISS vector index for fast similarity search
- Implemented full RAG: retrieve relevant chunks → augment prompt → generate grounded answer
- Measured retrieval quality with recall@k

RAG is the fastest way to give a model access to a document corpus. But it doesn't change how the model *reasons* or *writes*. For that, we need fine-tuning.

---

▶ **Next: [Notebook 04 — Fine-Tuning with LoRA](04_fine_tuning.ipynb)**